# Aura AI — Colab GPU Runtime
이 노트북은 Colab에서 Aura API와 Cloudflare Tunnel을 실행합니다. 런타임이 새로 시작되면 **런타임 → 모두 실행**을 누르면 환경을 다시 구성하고 서버를 재연결합니다.

Colab이 런타임 자체를 종료한 동안에는 어떤 코드도 실행될 수 없습니다. 새 런타임이 시작된 뒤 다시 실행해야 합니다.

Google Drive 인증이 실패하면 자동으로 런타임 임시 저장소를 사용합니다. 임시 저장소는 런타임 종료 시 삭제될 수 있습니다.

Drive mount가 실패하면 Google 계정 인증 후 Drive API 동기화를 시도합니다. 두 인증 방식이 모두 실패하면 임시 저장소를 사용합니다.

In [ ]:
import os, sys, subprocess, pathlib, time, threading, signal, shutil
REPO = '/content/Aura-AI'
if not os.path.isdir(os.path.join(REPO, '.git')):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Workshop-Koharu/Aura-AI.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=False)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'fastapi', 'uvicorn', 'python-multipart', 'pillow', 'psutil', 'requests', 'pypdf', 'python-docx', 'openpyxl', 'python-pptx', 'google-api-python-client'], check=True)
import torch, requests
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
from google.colab import drive, userdata
from getpass import getpass
DRIVE_CONNECTED = False
DRIVE_MODE = 'local-temporary'
DRIVE_DATA = pathlib.Path('/content/AuraAI-data')
try:
    drive.mount('/content/drive', force_remount=True)
    DRIVE_DATA = pathlib.Path('/content/drive/MyDrive/AuraAI/data')
    DRIVE_CONNECTED = True
    DRIVE_MODE = 'mounted-filesystem'
    print('Google Drive 파일 시스템 연결 완료')
except Exception as mount_error:
    print('Drive 파일 시스템 연결 실패:', type(mount_error).__name__, str(mount_error)[:180])
    try:
        from google.colab import auth
        auth.authenticate_user()
        import google.auth, hashlib, mimetypes
        from googleapiclient.discovery import build
        from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
        credentials, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive'])
        drive_service = build('drive', 'v3', credentials=credentials, cache_discovery=False)
        def drive_find_folder(name, parent_id):
            safe = name.replace("'", "\\\\'")
            q = f"name='{safe}' and mimeType='application/vnd.google-apps.folder' and '{parent_id}' in parents and trashed=false"
            found = drive_service.files().list(q=q, pageSize=100, fields='files(id,name)').execute().get('files', [])
            if found: return found[0]['id']
            return drive_service.files().create(body={'name':name,'mimeType':'application/vnd.google-apps.folder','parents':[parent_id]},fields='id').execute()['id']
        root_folder_id = drive_find_folder('AuraAI', 'root')
        data_folder_id = drive_find_folder('data', root_folder_id)
        def drive_list(parent_id):
            items, token = [], None
            while True:
                result = drive_service.files().list(q=f"'{parent_id}' in parents and trashed=false", pageSize=1000, pageToken=token, fields='nextPageToken,files(id,name,mimeType,md5Checksum,size)').execute()
                items.extend(result.get('files', [])); token=result.get('nextPageToken')
                if not token: return items
        def drive_download_tree(folder_id, local_dir):
            local_dir.mkdir(parents=True, exist_ok=True)
            for item in drive_list(folder_id):
                target=local_dir/item['name']
                if item['mimeType']=='application/vnd.google-apps.folder':
                    drive_download_tree(item['id'], target)
                elif not target.exists():
                    from googleapiclient.http import MediaIoBaseDownload
                    import io
                    request=drive_service.files().get_media(fileId=item['id']); stream=io.BytesIO(); downloader=MediaIoBaseDownload(stream,request)
                    done=False
                    while not done: _,done=downloader.next_chunk()
                    target.parent.mkdir(parents=True,exist_ok=True); target.write_bytes(stream.getvalue())
        def drive_upload_tree(local_dir, folder_id):
            remote={item['name']:item for item in drive_list(folder_id)}
            for local in local_dir.iterdir():
                if local.is_dir():
                    entry=remote.get(local.name)
                    child_id=entry['id'] if entry and entry['mimeType']=='application/vnd.google-apps.folder' else drive_service.files().create(body={'name':local.name,'mimeType':'application/vnd.google-apps.folder','parents':[folder_id]},fields='id').execute()['id']
                    drive_upload_tree(local,child_id)
                elif local.is_file() and not local.name.endswith('.tmp'):
                    digest=hashlib.md5(local.read_bytes()).hexdigest(); entry=remote.get(local.name)
                    if entry and entry.get('md5Checksum')==digest: continue
                    media=MediaFileUpload(str(local),mimetype=mimetypes.guess_type(local.name)[0] or 'application/octet-stream',resumable=True)
                    if entry: drive_service.files().update(fileId=entry['id'],media_body=media).execute()
                    else: drive_service.files().create(body={'name':local.name,'parents':[folder_id]},media_body=media).execute()
        DRIVE_CONNECTED=True
        DRIVE_MODE='drive-api-sync'
        print('Drive API 인증 성공. 파일 시스템 마운트 대신 Drive API 동기화를 사용합니다.')
    except Exception as api_error:
        print('Drive API 인증도 실패:', type(api_error).__name__, str(api_error)[:180])
        DRIVE_CONNECTED=False
DRIVE_DATA.mkdir(parents=True, exist_ok=True)
if DRIVE_MODE=='drive-api-sync':
    drive_download_tree(data_folder_id, DRIVE_DATA)
REPO_DATA = pathlib.Path(REPO) / 'data'
for name in ['text', 'image_text', 'audio_text', 'checkpoints', 'tokenizer']:
    src, dst = REPO_DATA / name, DRIVE_DATA / name
    dst.mkdir(parents=True, exist_ok=True)
    if src.is_symlink(): src.unlink()
    elif src.exists():
        for item in src.iterdir():
            target = dst / item.name
            if not target.exists(): shutil.copytree(item, target) if item.is_dir() else shutil.copy2(item, target)
        shutil.rmtree(src)
    if not src.exists(): src.symlink_to(dst, target_is_directory=True)
if DRIVE_MODE=='drive-api-sync':
    drive_upload_tree(DRIVE_DATA, data_folder_id)
    def drive_sync_worker():
        while True:
            time.sleep(180)
            try: drive_upload_tree(DRIVE_DATA, data_folder_id)
            except Exception as sync_error: print('Drive API sync retry:', type(sync_error).__name__, str(sync_error)[:120])
    threading.Thread(target=drive_sync_worker, daemon=True).start()
os.environ['AURA_DEVICE'] = 'cuda' if torch.cuda.is_available() else 'cpu'
os.environ['AURA_GPU_MEMORY_FRACTION'] = '0.33'
os.environ['AURA_CPU_THREADS'] = '2'
try:
    secret = userdata.get('AURA_GATEWAY_SECRET')
except Exception as secret_error:
    print('Colab Secrets 자동 읽기 실패:', type(secret_error).__name__)
    secret = ''
if not secret:
    secret = getpass('AURA_GATEWAY_SECRET 값을 입력하세요 (입력 내용은 화면에 표시되지 않습니다): ')
if not secret: raise RuntimeError('AURA_GATEWAY_SECRET 값이 비어 있습니다.')
os.environ['AURA_GATEWAY_SECRET'] = secret
print('Gateway secret loaded (hidden). Device:', os.environ['AURA_DEVICE'])
print('Drive persistence mode:', DRIVE_MODE)


## 학습 데이터
권한이 있는 UTF-8 TXT 파일만 업로드하세요. 파일은 Google Drive에 저장됩니다.

In [ ]:
from google.colab import files
uploaded = files.upload()
text_dir = DRIVE_DATA / 'text'
for name, content in uploaded.items():
    if name.lower().endswith('.txt'):
        (text_dir / pathlib.Path(name).name).write_bytes(content)
print('TXT count:', len(list(text_dir.glob('*.txt'))))


In [ ]:
from aura.training.runner import RUNNER
has_data = any((DRIVE_DATA / 'text').glob('*.txt')) or (DRIVE_DATA / 'image_text' / 'pairs.jsonl').exists() or (DRIVE_DATA / 'audio_text' / 'pairs.jsonl').exists()
if has_data: print('Training:', RUNNER.start())
else: print('학습 데이터가 없어 학습을 시작하지 않았습니다.')


In [ ]:
import subprocess, threading, requests, pathlib, re, urllib.request, os, time
API_LOG = open('/content/aura-api.log', 'a', buffering=1)
TUNNEL_LOG = open('/content/aura-tunnel.log', 'a', buffering=1)
CLOUDFLARED = '/usr/local/bin/cloudflared'
if not pathlib.Path(CLOUDFLARED).exists():
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', CLOUDFLARED)
    os.chmod(CLOUDFLARED, 0o755)
process_lock = threading.Lock()
processes = {'api': None, 'tunnel': None}
def start_api():
    p = processes['api']
    if p is None or p.poll() is not None:
        processes['api'] = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'api.main:app', '--host', '127.0.0.1', '--port', '8000'], cwd=REPO, env=os.environ.copy(), stdout=API_LOG, stderr=subprocess.STDOUT)
        print('Aura API process restarted.')
def wait_api():
    for _ in range(60):
        try:
            r = requests.get('http://127.0.0.1:8000/api/health', timeout=2)
            if r.ok: return True
        except requests.RequestException: pass
        time.sleep(2)
    return False
def start_tunnel():
    p = processes['tunnel']
    if p is None or p.poll() is not None:
        processes['tunnel'] = subprocess.Popen([CLOUDFLARED, 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'], stdout=TUNNEL_LOG, stderr=subprocess.STDOUT)
        print('Cloudflare Tunnel process started.')
def get_tunnel_url(timeout=180):
    log_path = pathlib.Path('/content/aura-tunnel.log')
    for _ in range(timeout // 2):
        try: log = log_path.read_text(errors='ignore')
        except OSError: log = ''
        matches = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', log)
        if matches: return matches[-1]
        time.sleep(2)
    return None
def watchdog():
    while True:
        with process_lock:
            start_api()
            if wait_api(): start_tunnel()
        time.sleep(15)
with process_lock:
    start_api()
if not wait_api(): raise RuntimeError('Aura API가 시작되지 않았습니다. /content/aura-api.log 확인')
with process_lock: start_tunnel()
threading.Thread(target=watchdog, daemon=True).start()
PUBLIC_URL = get_tunnel_url()
print('AURA_INFERENCE_URL:', PUBLIC_URL or 'URL 생성 실패 — /content/aura-tunnel.log 확인')
print('Vercel 프로젝트 환경변수 AURA_INFERENCE_URL에 위 URL을 등록하세요. URL은 런타임 재시작 시 바뀔 수 있습니다.')
print('Health:', requests.get('http://127.0.0.1:8000/api/health', timeout=5).text)



## 재접속 동작
- API 또는 Cloudflare Tunnel 프로세스가 중단되면 watchdog이 실행 중인 런타임 안에서 다시 시작합니다.
- Colab 런타임 자체가 종료되면 watchdog도 종료됩니다. 새 런타임에서 **런타임 → 모두 실행**을 실행하면 Drive 데이터와 체크포인트를 다시 연결하고 API/Tunnel을 재시작합니다.
- Quick Tunnel 주소는 새로 발급될 수 있으므로 Vercel의 `AURA_INFERENCE_URL`도 새 주소로 갱신해야 합니다.
- 상태 확인: `requests.get('http://127.0.0.1:8000/api/health').text`
